In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

In [3]:
df = pd.read_csv("creditcard1.csv")
print (df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


**PREPROCESSING**

In [4]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [5]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []

In [7]:
for train_idx, test_idx in skf.split(X_scaled, y):

    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [36]:
sm = SMOTE(sampling_strategy=0.2, random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

In [37]:
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))

# encoder
x = Dense(512, activation="relu", kernel_regularizer=l2(0.001))(input_layer)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)

x = Dense(128, activation="relu")(x)

x = Dense(64, activation="relu")(x)
x = Dense(32, activation="relu")(x)

latent = Dense(16, activation="relu")(x)

# decoder
x = Dense(32, activation="relu")(latent)
x = Dense(64, activation="relu")(x)
x = Dense(128, activation="relu")(x)
x = Dense(256, activation="relu")(x)
x = Dense(512, activation="relu")(x)

output = Dense(input_dim, activation="linear")(x)

autoencoder = Model(input_layer, output)

In [38]:
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="mse"
)

In [39]:
early = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [40]:
autoencoder.fit(
    X_train,
    X_train,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early, reduce_lr],
    verbose=1
)

Epoch 1/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 1.0770 - val_loss: 11.0977 - learning_rate: 1.0000e-04
Epoch 2/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.6572 - val_loss: 5.1498 - learning_rate: 1.0000e-04
Epoch 3/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.5450 - val_loss: 4.9688 - learning_rate: 1.0000e-04
Epoch 4/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.4577 - val_loss: 4.7683 - learning_rate: 1.0000e-04
Epoch 5/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.4009 - val_loss: 4.0470 - learning_rate: 1.0000e-04
Epoch 6/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.3679 - val_loss: 3.7831 - learning_rate: 1.0000e-04
Epoch 7/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.3394 - val_loss: 3.9558 - learning_rate: 1.0000e-04
Epoch 8/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.3138 - val_loss: 2.7481 - learning_rate: 1.0000e-04
Epoch 9/50
853/853 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.2966 - va

In [41]:
reconstructions = autoencoder.predict(X_test)

mse = np.mean(np.square(X_test - reconstructions), axis=1)

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step


In [43]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, mse)

optimal_idx = np.argmax(tpr - fpr)

best_threshold = thresholds[optimal_idx]

y_pred = (mse > best_threshold).astype(int)

In [44]:
y_pred = (mse > best_threshold).astype(int)

In [45]:
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, mse)

print("AUC:", auc)

auc_scores.append(auc)

              precision    recall  f1-score   support

           0       1.00      0.92      0.96     56863
           1       0.02      0.85      0.04        98

    accuracy                           0.92     56961
   macro avg       0.51      0.89      0.50     56961
weighted avg       1.00      0.92      0.96     56961

AUC: 0.9410654753081789


In [60]:
print("Mean AUC:", np.mean(auc_scores))

Mean AUC: 0.9374228139455844
